# 11 · Explorador documental híbrido — Memoria Chilena

Este notebook **no descarga PDFs ni usa OCR, widgets o claves de API**.
Lee automáticamente los resultados de las etapas 7, 8 y 10 de este proyecto.

**Qué sí hace:** búsqueda de fichas por **TF-IDF**, **embeddings MiniLM**, **fusión RRF** y **expansión RDF opcional**; muestra enlaces y los metadatos que fundamentan la recuperación.

**Qué no hace:** transcribir o interpretar el contenido integral de los documentos, ni redactar afirmaciones históricas mediante un LLM. Es el componente de **recuperación** que alimentará ese RAG.

Abre este cuaderno desde `notebooks/` y pulsa **Run → Run All Cells**. Las búsquedas de demostración usan resultados ya calculados, por lo que no requieren volver a descargar MiniLM. Las preguntas nuevas sí necesitan el modelo de la etapa 10 instalado o accesible.

## 1. Cargar resultados reales y comprobar consistencia

Se usan los identificadores MC del catálogo, no los nombres de archivos PDF. Las referencias RDF representan relaciones curatoriales entre fichas; **no equivalen a relaciones históricas**.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import io, json, zipfile, unicodedata, re, hashlib, shutil
import numpy as np
import pandas as pd

CWD=Path.cwd().resolve()
if CWD.name.lower()=='notebooks' and (CWD.parent/'data').is_dir():
    ROOT=CWD.parent
elif (CWD/'notebooks').is_dir() and (CWD/'data').is_dir():
    ROOT=CWD
else:
    raise FileNotFoundError('Abre el notebook en la carpeta notebooks/ del proyecto memoria_chilena')
OUT=ROOT/'data'/'output'; OUT.mkdir(parents=True,exist_ok=True)
SALIDA=OUT/'etapa11'; SALIDA.mkdir(parents=True,exist_ok=True)

def ubicar(n):
    for p in (OUT/f'etapa{n}.zip',ROOT/'data'/'input'/f'etapa{n}.zip'):
        if p.is_file(): return p
    raise FileNotFoundError(f'No encuentro etapa{n}.zip en data/output o data/input')

def leer_zip_csv(zfile,nombre):
    with zipfile.ZipFile(zfile) as z:
        return pd.read_csv(io.BytesIO(z.read(nombre)),dtype=str,keep_default_na=False)

Z7,Z8,Z10=ubicar(7),ubicar(8),ubicar(10)
catalogo=leer_zip_csv(Z7,'catalogo_para_busqueda.csv')
relaciones=leer_zip_csv(Z8,'relaciones_filtradas.csv')
rankings=leer_zip_csv(Z10,'rankings_cuatro_metodos.csv')
with zipfile.ZipFile(Z10) as z:
    emb_docs=np.load(io.BytesIO(z.read('embeddings_documentos.npy')),allow_pickle=False)
    meta=json.loads(z.read('embeddings_cache.json'))
    informe10=json.loads(z.read('informe_etapa10.json'))
ids=catalogo['mcid'].tolist()
assert len(ids)==len(set(ids))==10 and emb_docs.shape==(10,384)
assert set(ids)==set(rankings['mcid'])
assert len(rankings)==480 and rankings['pregunta'].nunique()==12
assert meta['modelo']=='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
assert np.isfinite(emb_docs).all() and np.allclose(np.linalg.norm(emb_docs,axis=1),1,atol=1e-3)
rankings['posicion']=rankings['posicion'].astype(int)
for c in ['score_metodo','score_tfidf','score_embedding_coseno','score_rrf','aporte_vecindad_rdf']:
    rankings[c]=rankings[c].astype(float)
print('Carpeta del proyecto:',ROOT)
print('Documentos:',len(ids),'| consultas evaluadas:',rankings.pregunta.nunique(), '| embeddings:',emb_docs.shape)
print('Juicios humanos completos:', informe10.get('etiquetas_validas', '?'),'/120')

Carpeta del proyecto: C:\Users\jvera\Documents\memoria_chilena
Documentos: 10 | consultas evaluadas: 12 | embeddings: (10, 384)
Juicios humanos completos: 0 /120


## 2. Consultar rankings existentes (sin cargar el modelo)

Estos resultados son exactamente los calculados en la etapa 10. Para una
pregunta nueva, la sección 4 calculará un embedding de la consulta con
el mismo modelo MiniLM.

In [2]:
METODOS=['A_tfidf','B_embeddings','C_hibrido_rrf','D_hibrido_rdf']
CONSULTAS_LISTAS=list(dict.fromkeys(rankings['pregunta']))
cat_id=catalogo.set_index('mcid')
LOG=[]


def explicar_catalogo(pregunta, mcid):
    r=cat_id.loc[mcid]
    términos=set(normalizar(pregunta).split())-{'que','obras','sobre','con','los','las','del','una','para','qué','estan','están','documentos','textos','fuentes','hay','relacionan','relacionados','acerca','materiales','se','en','de','el','la','al','y','un','a'}
    campos=[]
    for nombre,col in [('título','titulo'),('notas','notas'),('materias','materias'),('temas catalográficos','unidades')]:
        tokens=set(normalizar(r[col]).split())
        coincidencias=sorted(términos&tokens)
        if coincidencias: campos.append(f'{nombre}: {", ".join(coincidencias[:4])}')
    return ' | '.join(campos) or 'Similitud global del catálogo; ver ficha antes de interpretar'


def enriquecer(filas,pregunta):
    salida=filas.copy()
    salida['titulo']=salida.mcid.map(cat_id['titulo'].to_dict())
    salida['autor']=salida.mcid.map(cat_id['autor'].to_dict())
    salida['fecha_catalogo']=salida.mcid.map(cat_id['fecha'].to_dict())
    salida['ficha']=salida.mcid.map(cat_id['ficha'].to_dict())
    salida['pistas_metadatos']=[explicar_catalogo(pregunta,x) for x in salida['mcid']]
    return salida


def buscar_registrado(pregunta, metodo='C_hibrido_rrf', top_k=5):
    if metodo not in METODOS: raise ValueError('Método no reconocido: '+metodo)
    d=rankings[(rankings['pregunta']==pregunta)&(rankings['metodo']==metodo)].sort_values('posicion').head(top_k)
    if d.empty: raise KeyError('Pregunta no registrada. Usa explorar(...) con una pregunta nueva.')
    return enriquecer(d[['posicion','mcid','score_metodo']],pregunta)


def normalizar(s):
    s=unicodedata.normalize('NFKD',str(s).lower())
    s=''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'\s+',' ',re.sub(r'[^a-z0-9\s]',' ',s)).strip()


def explorar(pregunta, metodo='C_hibrido_rrf', top_k=5, guardar=True):
    if pregunta in CONSULTAS_LISTAS:
        resultado=buscar_registrado(pregunta,metodo,top_k)
    else:
        resultado=buscar_nuevo(pregunta,metodo,top_k)
    print('Pregunta:',pregunta)
    print('Método:',metodo)
    display(resultado[['posicion','titulo','mcid','pistas_metadatos','ficha']])
    print('NOTA: resultados del catálogo, NO afirmaciones sobre el texto íntegro de los libros.')
    if guardar:
        t=datetime.now(timezone.utc).isoformat()
        for r in resultado.itertuples(index=False):
            LOG.append({'fecha_utc':t,'pregunta':pregunta,'metodo':metodo,
                        'posicion':r.posicion,'mcid':r.mcid,'titulo':r.titulo,'ficha':r.ficha})
        exportar()
    return resultado

In [3]:
def exportar():
    df=pd.DataFrame(LOG)
    df.to_csv(SALIDA/'consultas_explorador.csv',index=False,encoding='utf-8-sig')
    manifest={
        'etapa':11,'generado_utc':datetime.now(timezone.utc).isoformat(),
        'documentos':len(ids),'consultas_demo':12,
        'consultas_guardadas':len(set(x['pregunta'] for x in LOG)),
        'metodo_inicial':'C_hibrido_rrf','alternativas':METODOS,
        'modelo_semantico':meta['modelo'],'embedding_dimension':384,
        'juicios_relevancia_validos':informe10.get('etiquetas_validas',0),
        'aviso':'Sólo evidencia catalográfica, no texto completo ni hechos históricos verificados.'
    }
    (SALIDA/'informe_etapa11.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    ruta=OUT/'etapa11.zip'
    with zipfile.ZipFile(ruta,'w',zipfile.ZIP_DEFLATED) as z:
        for f in sorted(SALIDA.iterdir()):
            if f.is_file(): z.write(f,f.name)
    return ruta

zip_final=exportar()
print('Archivo generado:',zip_final)

Archivo generado: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa11.zip


## 3. Consultas de demostración

La salida inicial no necesita conexión ni recálculo de embeddings. El método C
queda como opción inicial **por cautela**, no porque ya haya sido demostrado superior.

In [4]:
explorar('¿Qué textos coloniales están vinculados a la evangelización?',top_k=4)
explorar('¿Qué materiales hay acerca de idolatría en el Virreinato?',top_k=4)

Pregunta: ¿Qué textos coloniales están vinculados a la evangelización?
Método: C_hibrido_rrf


,posicion,titulo,mcid,pistas_metadatos,ficha
60,1,Doctrina christiana y catecismo para la instru...,MC0014753,Similitud global del catálogo; ver ficha antes...,https://www.memoriachilena.gob.cl/602/w3-artic...
61,2,Parte primera de la Chrónica del Perú : que tr...,MC0014754,Similitud global del catálogo; ver ficha antes...,https://www.memoriachilena.gob.cl/602/w3-artic...
62,3,Historia natural y moral de las Indias : en qu...,MC0014757,Similitud global del catálogo; ver ficha antes...,https://www.memoriachilena.gob.cl/602/w3-artic...
63,4,"Primera parte de los commentarios reales, que ...",MC0014755,Similitud global del catálogo; ver ficha antes...,https://www.memoriachilena.gob.cl/602/w3-artic...


NOTA: resultados del catálogo, NO afirmaciones sobre el texto íntegro de los libros.
Pregunta: ¿Qué materiales hay acerca de idolatría en el Virreinato?
Método: C_hibrido_rrf


,posicion,titulo,mcid,pistas_metadatos,ficha
300,1,El término Ylla y su potencial simbólico en el...,MC0038290,Similitud global del catálogo; ver ficha antes...,https://www.memoriachilena.gob.cl/602/w3-artic...
301,2,Extirpación de la idolatria del Piru,MC0014760,título: idolatria,https://www.memoriachilena.gob.cl/602/w3-artic...
302,3,"Primera parte de los commentarios reales, que ...",MC0014755,título: idolatria,https://www.memoriachilena.gob.cl/602/w3-artic...
303,4,Parte primera de la Chrónica del Perú : que tr...,MC0014754,Similitud global del catálogo; ver ficha antes...,https://www.memoriachilena.gob.cl/602/w3-artic...


NOTA: resultados del catálogo, NO afirmaciones sobre el texto íntegro de los libros.


,posicion,mcid,score_metodo,titulo,autor,fecha_catalogo,ficha,pistas_metadatos
300,1,MC0038290,0.032522,El término Ylla y su potencial simbólico en el...,"Manríquez Soto, Viviana",start=900;end=900,https://www.memoriachilena.gob.cl/602/w3-artic...,Similitud global del catálogo; ver ficha antes...
301,2,MC0014760,0.032266,Extirpación de la idolatria del Piru,"Arriaga, Pablo José de",start=1621;end=1621,https://www.memoriachilena.gob.cl/602/w3-artic...,título: idolatria
302,3,MC0014755,0.031754,"Primera parte de los commentarios reales, que ...","Vega, Garcilaso de la",start=1609;end=1609,https://www.memoriachilena.gob.cl/602/w3-artic...,título: idolatria
303,4,MC0014754,0.030579,Parte primera de la Chrónica del Perú : que tr...,"Cieza de León, Pedro de, ca. 1520-1554",1553,https://www.memoriachilena.gob.cl/602/w3-artic...,Similitud global del catálogo; ver ficha antes...


## 4. Preguntas nuevas (embeddings reales, sin OCR)

El siguiente código reconstruye el componente TF-IDF de la etapa 10 y usa
los vectores ya guardados para los documentos. Solo hay que calcular
**el embedding de la pregunta**; la primera consulta nueva carga MiniLM.

El RDF contribuye como **co-membresía catalográfica**, nunca como prueba de
hechos históricos. Los puntajes de los cuatro métodos no tienen una escala común.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
ALIASES={'quichua':'quechua','piru':'peru','yncas':'incas'}
UNIDAD_UNIVERSAL={'religiosidadandina'}
por_doc={r['mcid']:{x.strip() for x in r['unidades'].split('|') if x.strip()} for _,r in catalogo.iterrows()}

def expandir_alias(s):
    s=normalizar(s)
    for a,b in ALIASES.items(): s=re.sub(r'\b'+a+r'\b',b,s)
    return s

def texto_lexico(r):
    partes=[]
    for col,n in [('titulo',4),('autor',2),('materias',1),('notas',2),('datos',1)]:
        partes.extend([expandir_alias(r[col])]*n)
    partes.extend([expandir_alias(' '.join(sorted(por_doc[r['mcid']]-UNIDAD_UNIVERSAL)))]*2)
    return ' '.join(partes)

corpus=[texto_lexico(r) for _,r in catalogo.iterrows()]
vp=TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True,token_pattern=r'(?u)\b\w\w+\b')
vc=TfidfVectorizer(analyzer='char',ngram_range=(3,5),sublinear_tf=True)
Xp=vp.fit_transform(corpus); Xc=vc.fit_transform(corpus)

def score_lexico(pregunta):
    q=expandir_alias(pregunta)
    return .7*cosine_similarity(vp.transform([q]),Xp)[0]+.3*cosine_similarity(vc.transform([q]),Xc)[0]

def orden(scores): return sorted(range(len(ids)),key=lambda i:(-float(scores[i]),ids[i]))

def rrf(lex,sem,k=60):
    salida=np.zeros(len(ids))
    for idxs in ([i for i in orden(lex) if lex[i]>1e-12],orden(sem)):
        for p,i in enumerate(idxs,1): salida[i]+=1/(k+p)
    return salida

ix={mcid:i for i,mcid in enumerate(ids)}
A=np.zeros((10,10))
for r in relaciones.itertuples(index=False):
    i,j=ix[r.mcid_origen],ix[r.mcid_destino]
    A[i,j]=A[j,i]=max(A[i,j],float(r.peso_idf))
assert np.count_nonzero(np.triu(A,1))==14
sumas=A.sum(axis=1)
P=np.divide(A,sumas[:,None],out=np.zeros_like(A),where=sumas[:,None]>0)
MODELO_ID=meta['modelo']
_modelo=None

def modelo_consulta():
    global _modelo
    if _modelo is None:
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError as e:
            raise ImportError('Instala sentence-transformers en el kernel, igual que en la etapa 10, y repite la consulta.') from e
        _modelo=SentenceTransformer(MODELO_ID,device='cpu')
    return _modelo

def buscar_nuevo(pregunta,metodo='C_hibrido_rrf',top_k=5):
    if metodo not in METODOS: raise ValueError('Método desconocido')
    lex=score_lexico(pregunta)
    vec=np.asarray(modelo_consulta().encode([pregunta],convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=False),dtype=np.float32)[0]
    sem=vec@emb_docs.T
    fusion=rrf(lex,sem)
    norm=fusion/max(float(fusion.max()),1e-12)
    semilla=np.zeros(10)
    for i in orden(fusion)[:3]: semilla[i]=norm[i]
    rdf=norm+0.10*(P@semilla)
    scores=dict(zip(METODOS,[lex,sem,fusion,rdf]))[metodo]
    filas=pd.DataFrame([{'posicion':p,'mcid':ids[i],'score_metodo':float(scores[i])}
                        for p,i in enumerate(orden(scores)[:top_k],1)])
    return enriquecer(filas,pregunta)

print('El buscador está listo. Para una pregunta nueva, usa explorar("tu pregunta").')

El buscador está listo. Para una pregunta nueva, usa explorar("tu pregunta").


## 5. Probar tu propia pregunta

Modifica la variable `MI_PREGUNTA`. Si utilizas una de las doce preguntas
de la etapa 10, se reutilizan sus rankings guardados. Si escribes otra,
se ejecuta la recuperación semántica con el modelo MiniLM.

In [6]:
MI_PREGUNTA='¿Qué documentos permiten estudiar la geografía cultural del norte de Chile?'
METODO='C_hibrido_rrf'  # También: A_tfidf, B_embeddings, D_hibrido_rdf
explorar(MI_PREGUNTA,metodo=METODO,top_k=5)

Pregunta: ¿Qué documentos permiten estudiar la geografía cultural del norte de Chile?
Método: C_hibrido_rrf


,posicion,titulo,mcid,pistas_metadatos,ficha
180,1,La Tirana de Tarapacá,MC0023126,Similitud global del catálogo; ver ficha antes...,https://www.memoriachilena.gob.cl/602/w3-artic...
181,2,Con mi humilde devoción : bailes chinos en Chi...,MC0038283,título: chile,https://www.memoriachilena.gob.cl/602/w3-artic...
182,3,Historia natural y moral de las Indias : en qu...,MC0014757,Similitud global del catálogo; ver ficha antes...,https://www.memoriachilena.gob.cl/602/w3-artic...
183,4,Doctrina christiana y catecismo para la instru...,MC0014753,Similitud global del catálogo; ver ficha antes...,https://www.memoriachilena.gob.cl/602/w3-artic...
184,5,El Altiplano de Arica,MC0023128,Similitud global del catálogo; ver ficha antes...,https://www.memoriachilena.gob.cl/602/w3-artic...


NOTA: resultados del catálogo, NO afirmaciones sobre el texto íntegro de los libros.


,posicion,mcid,score_metodo,titulo,autor,fecha_catalogo,ficha,pistas_metadatos
180,1,MC0023126,0.032258,La Tirana de Tarapacá,"Uribe Echevarría, Juan, 1908-1988",start=1899;end=1999,https://www.memoriachilena.gob.cl/602/w3-artic...,Similitud global del catálogo; ver ficha antes...
181,2,MC0038283,0.031545,Con mi humilde devoción : bailes chinos en Chi...,"Mercado, Claudio | Rondón, Víctor",start=2003;end=2003,https://www.memoriachilena.gob.cl/602/w3-artic...,título: chile
182,3,MC0014757,0.031498,Historia natural y moral de las Indias : en qu...,"Acosta, José de",start=1590;end=1590,https://www.memoriachilena.gob.cl/602/w3-artic...,Similitud global del catálogo; ver ficha antes...
183,4,MC0014753,0.031010,Doctrina christiana y catecismo para la instru...,,1584,https://www.memoriachilena.gob.cl/602/w3-artic...,Similitud global del catálogo; ver ficha antes...
184,5,MC0023128,0.030886,El Altiplano de Arica,"Niemeyer Fernández, Hans, 1921-2005",start=1800;end=1999,https://www.memoriachilena.gob.cl/602/w3-artic...,Similitud global del catálogo; ver ficha antes...


## 6. Exportación automática y límites

Los resultados se guardan en `data/output/etapa11.zip` después de cada
consulta. Se conserva el modelo de evaluación de la etapa 10, sin inventar
etiquetas humanas ni métricas de calidad. Los enlaces apuntan a las fichas
originales de Memoria Chilena.

### Siguiente paso científico

Completar los 120 juicios de relevancia del notebook 09, comparando A/B/C/D con
nDCG@5 y Recall@5. Después podremos integrar una capa de respuesta con LLM
**limitada a afirmaciones presentes en las fichas** y con referencias verificables;
si queremos respuestas sobre lo que dicen los libros, necesitaremos evidencia
textual fiable de pasajes concretos.